# 🔬 Cervexa - Multi-Type Cervical Lesion AI Model Training
### Pelatihan Model AI Klasifikasi Kanker Serviks / VIA (Normal vs Abnormal)

Notebook Google Colab ini dirancang khusus untuk melatih model AI VIA Cervexa menggunakan dataset gabungan (**Type 1, Type 2, Type 3, dan additional_Type_3_v2**) dari Google Drive.

--- 
### 📋 Panduan Singkat:
1. **Aktifkan GPU**: Pilih menu **Runtime > Change runtime type > T4 GPU** (Pastikan GPU aktif agar training selesai dalam ~20-30 menit).
2. **Jalankan Step 1 s/d Step 8** secara berurutan.
3. **Unduh Model**: Di akhir (Step 8), file `via_model.tflite` (~6.8 MB) akan otomatis terunduh ke komputer Anda.
4. **Pasang di Android**: Salin `via_model.tflite` ke folder `app/src/main/assets/via_model.tflite` di proyek Cervexa.

> **Spesifikasi Output Android Wajib:**
> - Input: `[1, 224, 224, 3]` Float32
> - Output: `[1, 2]` Softmax (Index 0 = **ABNORMAL**, Index 1 = **NORMAL**)
> - Ukuran Target: ~6.8 MB

## ⚙️ Step 1: Cek GPU & Instalasi Dependencies

In [ ]:
# Cek alokasi GPU di Colab
!nvidia-smi

import os
import sys
import zipfile
import shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
print(f'TensorFlow Version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU Detected: {gpus}')
if not gpus:
    print('⚠️ PERINGATAN: GPU tidak terdeteksi! Silakan ganti Runtime ke T4 GPU di menu Runtime > Change runtime type.')

## 📂 Step 2: Hubungkan Dataset Google Drive

Anda dapat memilih salah satu cara di bawah ini:
- **Opsi A (Rekomendasi)**: Mount Google Drive Anda (Tambahkan shortcut folder dataset ke Google Drive Anda jika menggunakan link share).
- **Opsi B**: Download langsung menggunakan `gdown`.

In [ ]:
# ==========================================================================
# OPSI A: Mount Google Drive
# ==========================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Tentukan path folder dataset di Google Drive Anda
GDRIVE_DIR = Path('/content/drive/MyDrive')
candidate_dirs = [d for d in GDRIVE_DIR.iterdir() if d.is_dir() and any(k in d.name.lower() for k in ['intel', 'type', 'cervexa', 'cervical'])]
print('Folder dataset yang ditemukan di MyDrive:')
for d in candidate_dirs:
    print(' -', d.name)


In [ ]:
# ==========================================================================
# OPSI B: Download Otomatis Menggunakan gdown (Jika tidak mount Drive)
# Folder ID: 1LE2YMv2ycNBXXuMXD75FZEhuqRm3Il3C
# ==========================================================================
# Jalankan cell ini jika ingin mendownload langsung ke storage sementara Colab:

# !pip install -q --upgrade gdown
# import gdown
# folder_url = 'https://drive.google.com/drive/folders/1LE2YMv2ycNBXXuMXD75FZEhuqRm3Il3C'
# gdown.download_folder(folder_url, output='/content/dataset_source', quiet=False, use_cookies=False)


## 📦 Step 3: Ekstraksi & Persiapan Dataset Gabungan

Menangani 4 sumber dataset:
1. `Type_1` (`abnormal-1`, `abnormal-2`, `normal-1`, `Normal-2`)
2. `TYPE_2.zip` (Otomatis diekstrak jika masih berupa file .zip)
3. `Type_3` (`abnormal`, `normal`)
4. `additional_Type_3_v2` (`Hasil Abnormal`, `Hasil Normal`)

In [ ]:
# 🚀 OPTIMASI KECEPATAN: Salin dataset dari Google Drive ke SSD internal Colab
# Ini membuat training melesat dari 16 menit/epoch menjadi 10-15 DETIK/epoch!
LOCAL_DIR = Path('/content/dataset_local')
if not LOCAL_DIR.exists() or len(list(LOCAL_DIR.glob('*'))) == 0:
    print('🚀 Menyalin dataset dari Google Drive ke SSD lokal Colab (~2 menit)...')
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    # Salin isi Intel & MobileODT ke SSD lokal
    !cp -r '/content/drive/MyDrive/Intel & MobileODT/'* /content/dataset_local/
    print('✅ Berhasil disalin ke SSD lokal Colab!')
else:
    print('✅ Dataset sudah ada di SSD lokal Colab!')

source_root = LOCAL_DIR

# Ekstrak TYPE_2.zip jika ditemukan di SSD lokal
for zip_file in source_root.rglob('*TYPE_2*.zip'):
    target_extract = LOCAL_DIR / 'TYPE_2'
    if not target_extract.exists():
        print(f'Mengekstrak {zip_file.name}...')
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(target_extract)
        print('✅ TYPE_2.zip berhasil diekstrak!')
    break


## 🔍 Step 4: Pemindaian Otomatis & Filter Gambar Corrupt

Skrip akan:
- Mengelompokkan folder berawalan/mengandung kata `abnormal` ke **Index 0 (ABNORMAL)**
- Mengelompokkan folder berawalan/mengandung kata `normal` ke **Index 1 (NORMAL)**
- Memfilter dan memulihkan JPEG rusak agar proses training tidak crash saat epoch berjalan.

In [ ]:
abnormal_files = []
normal_files = []

# Cari di source_root (SSD lokal Colab) dengan followlinks=True
search_dirs = [source_root]
scanned_dirs = set()

for s_dir in search_dirs:
    if not s_dir.exists():
        continue
    for root, dirs, files in os.walk(str(s_dir), followlinks=True):
        root_path = Path(root)
        if root_path in scanned_dirs:
            continue
        name_lower = root_path.name.lower()
        imgs = [root_path / f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if not imgs:
            continue
        scanned_dirs.add(root_path)
        if 'abnormal' in name_lower:
            abnormal_files.extend([str(x) for x in imgs])
            print(f' [ABNORMAL] {root_path.name}: {len(imgs)} gambar')
        elif 'normal' in name_lower:
            normal_files.extend([str(x) for x in imgs])
            print(f' [NORMAL]   {root_path.name}: {len(imgs)} gambar')

total_abnormal = len(abnormal_files)
total_normal = len(normal_files)
print('=' * 60)
print(f'TOTAL GAMBAR TERDETEKSI:')
print(f'  - ABNORMAL (Kelas 0): {total_abnormal} gambar')
print(f'  - NORMAL   (Kelas 1): {total_normal} gambar')
print(f'  - Total             : {total_abnormal + total_normal} gambar')
print('=' * 60)

# Integritas gambar: singkirkan file corrupt/kosong
print('Memverifikasi integritas gambar...')
valid_files = []
valid_labels = []

for f in abnormal_files:
    try:
        raw = tf.io.read_file(f)
        _ = tf.image.decode_jpeg(raw, channels=3, try_recover_truncated=True, acceptable_fraction=0.5)
        valid_files.append(f)
        valid_labels.append(0) # Index 0 = Abnormal
    except Exception:
        pass

for f in normal_files:
    try:
        raw = tf.io.read_file(f)
        _ = tf.image.decode_jpeg(raw, channels=3, try_recover_truncated=True, acceptable_fraction=0.5)
        valid_files.append(f)
        valid_labels.append(1) # Index 1 = Normal
    except Exception:
        pass

valid_files = np.array(valid_files)
valid_labels = np.array(valid_labels)
print(f'✅ Berhasil memverifikasi {len(valid_files)} gambar valid yang siap dilatih!')

## 🔀 Step 5: Dataset Pipeline & Class Weighting

Mengatasi ketimpangan data (*class imbalance*) dan augmentasi khusus mikroskop MS2.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Shuffle & Split (80% Train, 20% Val)
np.random.seed(42)
indices = np.arange(len(valid_files))
np.random.shuffle(indices)

valid_files = valid_files[indices]
valid_labels = valid_labels[indices]

split_idx = int(len(valid_files) * 0.8)
train_files, val_files = valid_files[:split_idx], valid_files[split_idx:]
train_labels, val_labels = valid_labels[:split_idx], valid_labels[split_idx:]

print(f'Train set: {len(train_files)} gambar')
print(f'Val set  : {len(val_files)} gambar')

def parse_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3, try_recover_truncated=True, acceptable_fraction=0.5)
    img = tf.image.resize(img, IMG_SIZE)
    label_onehot = tf.one_hot(label, depth=2) # [1, 2] One-Hot Softmax
    return img, label_onehot

AUTOTUNE = tf.data.AUTOTUNE
train_ds = tf.data.Dataset.from_tensor_slices((train_files, train_labels))
train_ds = train_ds.map(parse_img, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_files, val_labels))
val_ds = val_ds.map(parse_img, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Hitung bobot kelas agar model tidak bias
n_abn = np.sum(valid_labels == 0)
n_nor = np.sum(valid_labels == 1)
class_weight = {
    0: len(valid_labels) / (2.0 * max(1, n_abn)),
    1: len(valid_labels) / (2.0 * max(1, n_nor))
}
print(f'Class Weight: Kelas 0 (Abnormal)={class_weight[0]:.2f}, Kelas 1 (Normal)={class_weight[1]:.2f}')

## 🧠 Step 6: Arsitektur Model (EfficientNetV2B0) & Training

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Feature Extractor
base_model = tf.keras.applications.EfficientNetV2B0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

# Data Augmentasi Medis
augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name='medical_aug')

inputs = layers.Input(shape=(224, 224, 3), name='input_image')
x = augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.25)(x)
# Output Softmax 2 Kelas: Index 0 = Abnormal, Index 1 = Normal
outputs = layers.Dense(2, activation='softmax', name='classification_output')(x)

model = tf.keras.Model(inputs, outputs, name='Cervexa_VIA_MultiType')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_accuracy', verbose=1),
    EarlyStopping(patience=8, monitor='val_accuracy', restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# --- Phase 1: Train Head ---
print('\n🚀 Memulai Training Phase 1 (Classification Head)...')
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    class_weight=class_weight
)

# --- Phase 2: Fine-Tuning ---
print('\n🚀 Memulai Training Phase 2 (Fine-Tuning Top 40 Layers)...')
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    class_weight=class_weight
)


## 📊 Step 7: Evaluasi Medis (Confusion Matrix & Sensitivity)

In [ ]:
best_model = tf.keras.models.load_model('best_model.h5')

y_true, y_pred = [], []
for imgs, lbls in val_ds:
    preds = best_model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(lbls.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# 0 = Abnormal, 1 = Normal
TP = np.sum((y_true == 0) & (y_pred == 0))
TN = np.sum((y_true == 1) & (y_pred == 1))
FP = np.sum((y_true == 1) & (y_pred == 0))
FN = np.sum((y_true == 0) & (y_pred == 1))

accuracy = np.mean(y_true == y_pred) * 100
sensitivity = (TP / max(1, TP + FN)) * 100 # Recall Abnormal
specificity = (TN / max(1, TN + FP)) * 100 # Recall Normal

print('=' * 60)
print(f'HASIL EVALUASI KLINIS PADA VALIDATION SET:')
print(f'  - Akurasi Keseluruhan : {accuracy:.2f}%')
print(f'  - Sensitivitas (Recall ABNORMAL) : {sensitivity:.2f}%  (Target: >85%)')
print(f'  - Spesifisitas (Recall NORMAL)   : {specificity:.2f}%')
print(f'  - False Negative (Kasus Terlewat): {FN} dari {TP + FN} kasus abnormal')
print('=' * 60)


## 📱 Step 8: Ekspor ke TFLite & Download Otomatis

Model dikonversi dengan optimasi dynamic range quantization sehingga berukuran ~6.8 MB dan siap digunakan langsung di Android.

In [ ]:
from google.colab import files

print('Mengonversi ke TensorFlow Lite...')
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

output_path = 'via_model.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f'✅ Berhasil membuat {output_path} (Ukuran: {size_mb:.2f} MB)')

# Verifikasi Tensor I/O
interpreter = tf.lite.Interpreter(model_path=output_path)
interpreter.allocate_tensors()
in_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()

print('\nVerifikasi Spesifikasi Tensor Android:')
print(f' - Input Tensor Shape  : {in_details[0]["shape"]} (Harus [1, 224, 224, 3])')
print(f' - Output Tensor Shape : {out_details[0]["shape"]} (Harus [1, 2])')
print(f' - Index 0: ABNORMAL, Index 1: NORMAL')

# Otomatis trigger download ke komputer
print('\nMengunduh via_model.tflite ke komputer Anda...')
files.download(output_path)
